# 🧪 Clinical Trial Assistant — Semantic Chatbot Demo

This notebook demonstrates a Retrieval-Augmented Generation (RAG) system for a **Clinical Trial Assistant**.  
The assistant answers user questions about clinical trials using:

- 📄 A predefined knowledge base
- 🧠 Sentence embeddings for semantic search
- 🔎 Cosine similarity for retrieval
- 🤖 A Large Language Model fallback for unmatched queries

The goal is to simulate how an AI assistant can support patients and staff in understanding clinical trial processes.



## 📁 Project Structure

```text
clinical-trial-assistant/
├── data/
│   ├── knowledge_base.csv
│   ├── chatbot_responses.json
│   ├── processed_queries.csv
│   ├── query_responses.json
│   ├── predefined_responses.json
├── notebooks/
│   └── 02_clinical_trial_assistant_demo.ipynb
├── src/
│   ├── embedder.py
│   ├── retriever.py
│   ├── generator.py
│   ├── chatbot.py


## ⚙️ Imports & Setup (Code)

In [1]:
# Install dependencies (Colab / local)
!pip install -q sentence-transformers groq python-dotenv pandas

# Silence logs
import os, warnings
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

# Load Groq API key (Colab)
from google.colab import userdata
import os
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
print("API key loaded:", bool(os.environ.get("GROQ_API_KEY")))

# Setup path
import sys

PROJECT_ROOT = "/content/clinical-trial-assistant"

# Add project root to Python path
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# Make sure folders are packages
open(os.path.join(PROJECT_ROOT, "__init__.py"), "w").close()
open(os.path.join(PROJECT_ROOT, "src", "__init__.py"), "w").close()

print("✅ Project path configured")

API key loaded: True
✅ Project path configured


# Load & Inspect Knowledge Base

In [4]:
import os
import pandas as pd

# Load the dataset
df = pd.read_csv("clinical-trial-assistant/data/knowledge_base.csv")

# Display the first few entries
df.head()

,document_id,document_text,metadata
0,1,This document provides details about participa...,Requires payment information
1,2,This document provides details about study rul...,Requires protocol compliance
2,3,This document provides details about informed ...,Requires signed consent
3,4,This document provides details about eligibili...,Requires eligibility verification
4,5,This document provides details about adverse e...,Requires safety reporting


# Build knowledge embeddings

*   Load the dataset (knowledge_base.csv).
*   Generate text embeddings. Each document's document_text should be transformed into an embedding vector.
*   Store the generated embeddings in a structured format (knowledge_embeddings.json) with the following format available below.
*   Store the embedded data and associated metadata for retrieval.

In [5]:
import os
import json
import pandas as pd
from src.embedder import Embedder

DATA_PATH = "/content/clinical-trial-assistant/data/knowledge_base.csv"
OUTPUT_PATH = "/content/clinical-trial-assistant/data/knowledge_embeddings.json"

TEXT_COLUMN = "document_text"  # change if your CSV uses a different name


def main():
    print("📥 Loading dataset...")
    df = pd.read_csv(DATA_PATH)

    if TEXT_COLUMN not in df.columns:
        raise ValueError(f"Column '{TEXT_COLUMN}' not found in CSV.")

    texts = df[TEXT_COLUMN].fillna("").tolist()

    print(f"🔎 Found {len(texts)} documents to embed.")

    embedder = Embedder()
    print("🧠 Generating embeddings...")
    embeddings = embedder.embed(texts, batch_size=32, retries=3)

    print("💾 Saving embeddings to JSON...")
    records = []
    for i, (text, emb) in enumerate(zip(texts, embeddings)):
        record = {
            "id": f"doc_{i}",
            "text": text,
            "embedding": emb.tolist() if hasattr(emb, "tolist") else emb,
            "metadata": {
                "source": "knowledge_base.csv",
                "row_index": i
            }
        }
        records.append(record)

    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

    print(f"✅ Done. Saved {len(records)} embeddings to {OUTPUT_PATH}")


if __name__ == "__main__":
    main()

📥 Loading dataset...
🔎 Found 100 documents to embed.
🧠 Generating embeddings...
💾 Saving embeddings to JSON...
✅ Done. Saved 100 embeddings to /content/clinical-trial-assistant/data/knowledge_embeddings.json


# Similarity Search on predefined responses


*   Load the dataset (processed_queries.csv).
*   Retrieve responses by using cosine similarity to perform a similarity search against predefined responses in predefined_responses.json.
*   Compute confidence scores for retrieved responses, scaled to 0-1.
*   Store the structured responses in a JSON file (query_responses.json), suitable for integration with other applications.



In [6]:
import json
import pandas as pd
from src.retriever import Retriever
from src.generator import Generator

INPUT_QUERIES = "/content/clinical-trial-assistant/data/processed_queries.csv"
OUTPUT_JSON = "/content/clinical-trial-assistant/data/query_responses.json"


def main():
    print("📥 Loading queries...")
    df = pd.read_csv(INPUT_QUERIES)

    retriever = Retriever()
    generator = Generator()

    results = []

    for idx, row in df.iterrows():
        query_id = int(row["query_id"])
        query_text = row["query_text"]

        print(f"🔎 Query {query_id}: {query_text}")

        top_responses, confidence_scores = retriever.retrieve(query_text, top_k=3)

        structured = {
            "query_id": query_id,
            "query_text": query_text,
            "top_responses": top_responses,
            "confidence_scores": [round(c, 3) for c in confidence_scores]
        }

        results.append(structured)

    print("💾 Saving structured responses...")
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print(f"✅ Done. Saved {len(results)} records to {OUTPUT_JSON}")


if __name__ == "__main__":
    main()

📥 Loading queries...


🔎 Query 1: When is my screening visit?
🔎 Query 2: What happens during the screening appointment?
🔎 Query 3: What do I need to bring to my first visit?
🔎 Query 4: How long does Visit 1 usually take?
🔎 Query 5: What happens on Day 0 of the study?
🔎 Query 6: How many visits are there in total?
🔎 Query 7: Can I reschedule a study visit?
🔎 Query 8: What if I miss a scheduled appointment?
🔎 Query 9: Do I have to attend every follow-up visit?
🔎 Query 10: How often will I come to the clinic?
🔎 Query 11: Will I receive payment for participating?
🔎 Query 12: When will I get reimbursed for travel?
🔎 Query 13: How much compensation do participants receive?
🔎 Query 14: How do I provide my payment details?
🔎 Query 15: Are meals or transportation covered?
🔎 Query 16: Can I withdraw from the study at any time?
🔎 Query 17: What happens if I decide to leave the study early?
🔎 Query 18: Will leaving the study affect my medical care?
🔎 Query 19: Do I need to give a reason to withdraw?
🔎 Query 20: How do I

# ChatBot Demo

**Features**
*   Accepts customer queries via text input.
*   Search for the most relevant responses from a predefined set of responses (chatbot_responses.json).
*   Compute semantic similarity between queries. If no relevant response is found from the predefined set, generates a new response.
*   Stores conversation history, including:

Query text,
Retrieved response,
Timestamp of the interaction,
Confidence score of the response.




In [7]:
from src.chatbot import Chatbot

def main():
    bot = Chatbot()

    test_queries = [
        "Am I allowed to take supplements?",                         # exact
        "When can I talk to someone from support?",                  # paraphrase
        "What is your refund policy?"                                # open-ended (not predefined)
    ]

    for q in test_queries:
        print(f"\n👤 User: {q}")
        result = bot.get_response(q)
        print(f"🤖 Bot: {result['retrieved_response']} (conf={result['confidence_score']})")

    bot.save_history()
    print("\n💾 Saved to data/sample_chatbot_responses.json")


if __name__ == "__main__":
    main()



👤 User: Am I allowed to take supplements?
🤖 Bot: Some supplements are not allowed. Always check with the study team before taking any vitamins or herbal products. (conf=0.79)

👤 User: When can I talk to someone from support?
🤖 Bot: As a clinical trial assistant, I'm happy to help with any questions or concerns you may have. 

If you need to speak with someone from support, you can reach out to us during the following hours:

- Monday to Friday: 8:00 AM to 5:00 PM (your local time)
- We also have an after-hours support line available for urgent matters. Please call our main number and follow the prompts to reach the on-call support team.

You can also reach out to us via email at [support@clinicaltrial.com](mailto:support@clinicaltrial.com). We'll respond to your email within 24 hours.

If you have any questions or concerns outside of these hours, please don't hesitate to reach out. We're here to help. (conf=0.3)

👤 User: What is your refund policy?
🤖 Bot: As a clinical trial assistant